In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

In [2]:
# cms_spread_options_single_look = [
#     "QZQHDJR8ZR2C",
# 	"QZSP0NTLKKFJ",
# 	"QZP43JLWTM5W",
# 	"QZVH3T5N6GJ9",
# 	"QZD8FJ9CGMZ2",
# 	"QZH64P6BDDFN",
# 	"QZKC9F9FV3ZV",
# ]
# df[df["UPI Underlier Name"] == "USD-SOFR ICE Swap Rate vs USD-SOFR-COMPOUND"].to_csv("USD-SOFR ICE Swap Rate vs USD-SOFR-COMPOUND sdr trades.csv")

In [3]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from SDRUtils.products.usd.sofr_swaps import USD_SOFR_SwapProduct 
from SDRUtils.products.usd.usd_swaptions import USD_Swaptions

In [4]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

# as_of = datetime.date(2026, 2, 23)
# start = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 0, 0))
# end = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 23, 59))

start = NY_tz.localize(datetime.datetime(2026, 2, 26, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 2, 26, 23, 59))

# mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
# pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
df

MERGING SLICES...: 100%|██████████| 2/2 [00:00<00:00, 80.18it/s]


,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price currency,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name
0,2183056930000000201,105822307,TERM,ETRM,2026-02-26 05:00:00+00:00,None,IR,None,N,False,...,,NaN,,,NaN,None,None,QZGZ99NL7HC2,NA/Swaps Oth Nstd,USD-SOFR
1,2168267888000000101,2168125577000000301,EROR,,2026-02-26 05:00:22+00:00,None,IR,None,I,True,...,,NaN,,,NaN,None,None,QZ1CXH05JJJH,NA/Swap Fxd Flt USD,USD-SOFR-COMPOUND
2,2168267889000000201,,NEWT,TRAD,2026-02-26 05:00:22+00:00,None,IR,None,I,True,...,,NaN,,,NaN,None,None,QZ1CXH05JJJH,NA/Swap Fxd Flt USD,USD-SOFR-COMPOUND
3,2168322401000000101,,NEWT,TRAD,2026-02-26 05:00:25+00:00,None,IR,None,N,False,...,,NaN,,,NaN,None,None,QZXF4PFZSLP7,NA/Swap OIS INR,INR-MIBOR-OIS Compound
4,2168259699000000101,,NEWT,TRAD,2026-02-26 05:00:38+00:00,None,IR,None,I,True,...,,NaN,,,NaN,None,None,QZV61TRS4HD9,NA/Swap OIS JPY,JPY-TONA-OIS-COMPOUND
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23603,2184628779000000101,,NEWT,TRAD,2026-02-27 04:57:31+00:00,None,IR,None,I,False,...,,NaN,,,NaN,None,None,QZPB1RXQ05RR,NA/Swap OIS INR,INR-MIBOR-OIS-COMPOUND
23604,2184628991000000101,,NEWT,TRAD,2026-02-27 04:57:53+00:00,None,IR,None,I,True,...,,NaN,,,NaN,None,None,QZV61TRS4HD9,NA/Swap OIS JPY,JPY-TONA-OIS-COMPOUND
23605,2184630470000000101,,NEWT,TRAD,2026-02-27 04:57:58+00:00,None,IR,None,I,True,...,,NaN,,,NaN,None,None,QZ1CXH05JJJH,NA/Swap Fxd Flt USD,USD-SOFR-COMPOUND
23606,2184629199000000101,,NEWT,TRAD,2026-02-27 04:58:09+00:00,False,IR,None,I,True,...,,NaN,,,NaN,None,None,QZ5M5RFTF88M,NA/Swap OIS AUD,AUD-AONIA-OIS-COMPOUND


In [10]:
df[
    (df["UPI Underlier Name"].str.lower().str.contains("vs"))
    & (df["UPI Underlier Name"].str.lower().str.contains("usd"))
    & (df["UPI Underlier Name"].str.lower().str.contains("cad"))
].to_csv("02-26-2026-usdcad-xccy-basis-sdr.csv")
# ["UPI Underlier Name"].value_counts()

In [221]:
start = NY_tz.localize(datetime.datetime(2026, 2, 26, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 2, 26, 23, 59))

# sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path, merge_package_legs=False)
sdf = USD_SOFR_SwapProduct().build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=False, merge_package_legs=False)

Classifying Trades:  42%|████▏     | 1688/3984 [00:10<00:11, 191.98trade/s]ERROR	Task(Task-2) SDRUtils.products.base:base.py:classify_messages()- Failed to classify trade 2179021137000002901: ValueError: A Schedule could not be generated from the parameter combinations.
ERROR	Task(Task-2) SDRUtils.products.base:base.py:classify_messages()- Failed to classify trade 2179021138000003001: ValueError: A Schedule could not be generated from the parameter combinations.
ERROR	Task(Task-2) SDRUtils.products.base:base.py:classify_messages()- Failed to classify trade 2179021140000003201: ValueError: A Schedule could not be generated from the parameter combinations.
ERROR	Task(Task-2) SDRUtils.products.base:base.py:classify_messages()- Failed to classify trade 2179021139000003101: ValueError: A Schedule could not be generated from the parameter combinations.
FETCHING DELIVERY BASKETS...: 100%|██████████| 6/6 [00:16<00:00,  2.81s/it]


In [224]:
sdf["trade_label"].value_counts()

trade_label
spot 10Y           451
spot 5Y            369
spot 30Y           343
spot 2Y            192
spot 7Y            151
                  ... 
10M 7Y               1
8M 2Y                1
20Y1M IMM_Z2055      1
1Y10M 30Y            1
5D 8Y                1
Name: count, Length: 363, dtype: int64